# NB04 — CLI Tools with Click and argparse

**Module 16 — Research Software Engineering**

---

## Motivation

Bioinformatics pipelines are run from the command line — not from a Jupyter notebook, not from a Python REPL. A function that only lives inside a notebook is not a reusable tool. A command-line interface (CLI) makes your code composable: it can be called from bash scripts, snakemake rules, nextflow processes, and CI pipelines. `compbio gc --sequence ATCGATCG` is a shareable, reproducible unit of computation.

## Intuition

Click uses Python decorators to turn ordinary functions into CLI commands. You write normal Python, add `@click.command()` and `@click.option()` decorators, and Click handles parsing, help text, and error messages automatically. Click also provides `CliRunner` for testing CLI commands programmatically — no subprocess needed.

## Biological Background

The `compbio` CLI we build wraps three operations that a bioinformatician uses dozens of times per week: `gc` (GC content, essential for primer/probe design and GC-bias correction), `revcomp` (reverse complement, used in primer design and for reading reverse-strand annotations), and `fasta-stats` (quick summary statistics on a FASTA file before downstream processing).

---

In [ ]:
# Click vs argparse: a side-by-side comparison
import textwrap

argparse_example = textwrap.dedent("""
    # argparse version
    import argparse

    parser = argparse.ArgumentParser(description='Compute GC content')
    parser.add_argument('--sequence', required=True, help='DNA sequence')
    args = parser.parse_args()
    seq = args.sequence.upper()
    gc = (seq.count('G') + seq.count('C')) / len(seq)
    print(f'GC content: {gc:.2%}')
""")

click_example = textwrap.dedent("""
    # Click version
    import click

    @click.command()
    @click.option('--sequence', required=True, help='DNA sequence')
    def gc(sequence):
        \"\"\"Compute GC content of a DNA sequence.\"\"\"
        seq = sequence.upper()
        gc_val = (seq.count('G') + seq.count('C')) / len(seq)
        click.echo(f'GC content: {gc_val:.2%}')

    if __name__ == '__main__':
        gc()
""")

print('argparse approach:')
print(argparse_example)
print('Click approach:')
print(click_example)

In [ ]:
try:
    import click
    from click.testing import CliRunner
    HAS_CLICK = True
except ImportError:
    HAS_CLICK = False
    print('Click not installed. Install with: pip install click')
    print('Showing code structure only.')

# Stub implementations (same as NB02)
def reverse_complement(seq: str) -> str:
    seq = seq.upper()
    valid = set('ATCG')
    if set(seq) - valid:
        raise ValueError(f'Invalid nucleotides: {set(seq) - valid}')
    return seq.translate(str.maketrans('ATCG', 'TAGC'))[::-1]

def gc_content(seq: str) -> float:
    if not seq:
        return 0.0
    seq = seq.upper()
    return (seq.count('G') + seq.count('C')) / len(seq)

print(f'Click available: {HAS_CLICK}')

In [ ]:
# Build the compbio CLI
if HAS_CLICK:
    @click.group()
    @click.version_option(version='0.1.0', prog_name='compbio')
    def cli():
        """compbio — computational biology utilities CLI."""
        pass


    @cli.command()
    @click.option('--sequence', '-s', required=True,
                  help='DNA sequence (A, T, C, G only).')
    @click.option('--percent/--fraction', default=True,
                  help='Output as percentage (default) or fraction.')
    def gc(sequence: str, percent: bool):
        """Compute GC content of a DNA sequence."""
        try:
            result = gc_content(sequence)
        except ValueError as e:
            raise click.BadParameter(str(e), param_hint='--sequence')
        if percent:
            click.echo(f'GC content: {result:.2%}')
        else:
            click.echo(f'GC content: {result:.4f}')


    @cli.command()
    @click.option('--sequence', '-s', required=True,
                  help='DNA sequence to reverse-complement.')
    def revcomp(sequence: str):
        """Compute the reverse complement of a DNA sequence."""
        try:
            result = reverse_complement(sequence)
        except ValueError as e:
            raise click.BadParameter(str(e), param_hint='--sequence')
        click.echo(result)


    @cli.command('fasta-stats')
    @click.argument('file', type=click.Path(exists=False))
    @click.option('--verbose', '-v', is_flag=True, help='Show per-sequence stats.')
    def fasta_stats(file: str, verbose: bool):
        """Print summary statistics for a FASTA file."""
        # Full implementation in NB10; stub here
        click.echo(f'fasta-stats: would parse {file}')
        if verbose:
            click.echo('(verbose mode: per-sequence stats)')

    print('CLI defined: compbio gc, compbio revcomp, compbio fasta-stats')

In [ ]:
# Test the CLI with CliRunner (no subprocess needed)
if HAS_CLICK:
    runner = CliRunner()

    test_cases = [
        (['gc', '--sequence', 'ATCG'], 'GC content: 50.00%'),
        (['gc', '--sequence', 'AAAA'], 'GC content: 0.00%'),
        (['gc', '--sequence', 'CCCC'], 'GC content: 100.00%'),
        (['revcomp', '--sequence', 'ATCG'], 'CGAT'),
        (['revcomp', '--sequence', 'AAAA'], 'TTTT'),
    ]

    print('CLI test results:')
    for args, expected_substr in test_cases:
        result = runner.invoke(cli, args)
        ok = result.exit_code == 0 and expected_substr in result.output
        status = 'PASS' if ok else 'FAIL'
        print(f'  [{status}] compbio {" ".join(args)}')
        if not ok:
            print(f'         Output: {result.output!r}')
            print(f'         Expected substring: {expected_substr!r}')
else:
    print('Click not available — install and re-run.')

## Visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('CLI Architecture', fontsize=14, fontweight='bold')

# --- Panel 1: Command tree ---
ax1 = axes[0]
ax1.set_xlim(0, 10)
ax1.set_ylim(0, 10)
ax1.axis('off')
ax1.set_title('compbio Command Tree', fontweight='bold')

# Root
root_box = mpatches.FancyBboxPatch((3.5, 8.5), 3, 0.8, boxstyle='round,pad=0.1',
                                     facecolor='#2c3e50', edgecolor='white')
ax1.add_patch(root_box)
ax1.text(5, 8.9, 'compbio', ha='center', va='center', color='white',
         fontweight='bold', fontsize=12, fontfamily='monospace')

commands = [
    (2, 6.5, 'gc', '--sequence\n--percent/--fraction', '#3498db'),
    (5, 6.5, 'revcomp', '--sequence', '#8e44ad'),
    (8, 6.5, 'fasta-stats', 'FILE\n--verbose', '#27ae60'),
]

for cx, cy, name, opts, color in commands:
    ax1.plot([5, cx], [8.5, cy + 0.4], 'k-', alpha=0.4, lw=1.5)
    cmd_box = mpatches.FancyBboxPatch((cx - 1.2, cy - 0.3), 2.4, 0.8,
                                       boxstyle='round,pad=0.1',
                                       facecolor=color, edgecolor='white')
    ax1.add_patch(cmd_box)
    ax1.text(cx, cy + 0.1, name, ha='center', va='center', color='white',
             fontweight='bold', fontsize=10, fontfamily='monospace')
    ax1.text(cx, cy - 1.0, opts, ha='center', va='center', fontsize=8,
             color='#555', fontfamily='monospace')

# --- Panel 2: Click option/argument flow ---
ax2 = axes[1]
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 10)
ax2.axis('off')
ax2.set_title('Click Decorator Flow', fontweight='bold')

flow_steps = [
    (5, 9.0, 'User types:\ncompbio gc --sequence ATCG', '#2c3e50'),
    (5, 7.0, '@click.group()\nregisters top-level group', '#3498db'),
    (5, 5.0, '@click.command()\nregisters subcommand', '#8e44ad'),
    (5, 3.0, '@click.option()\nparses --sequence', '#e67e22'),
    (5, 1.0, 'gc(sequence="ATCG")\ncalled with parsed args', '#27ae60'),
]

for x, y, label, color in flow_steps:
    bbox = mpatches.FancyBboxPatch((x - 3, y - 0.5), 6, 1.0,
                                    boxstyle='round,pad=0.1',
                                    facecolor=color, alpha=0.85, edgecolor='white')
    ax2.add_patch(bbox)
    ax2.text(x, y, label, ha='center', va='center', fontsize=8.5,
             color='white', fontweight='bold')

for i in range(len(flow_steps) - 1):
    _, y1, _, _ = flow_steps[i]
    _, y2, _, _ = flow_steps[i + 1]
    ax2.annotate('', xy=(5, y2 + 0.5), xytext=(5, y1 - 0.5),
                 arrowprops=dict(arrowstyle='->', color='#555', lw=2))

plt.tight_layout()
plt.savefig('cli_architecture.png', dpi=120, bbox_inches='tight')
plt.show()

## Exercises

See `exercises/README.md` → E04.

1. Add a `compbio tm --sequence ATCGATCG` command that computes melting temperature using the Wallace rule from NB03.
2. Test it with `CliRunner`: write 5 test cases.
3. What is the difference between `@click.option` and `@click.argument`? When would you use each?

## Quiz

1. What does `@click.group()` do?
2. How do you test a Click CLI without running a subprocess?
3. What is the difference between `--flag` and `--flag/--no-flag` in Click?
4. How does Click handle type validation (e.g., ensuring a value is an integer)?
5. How do you add a `--version` flag to a Click CLI?

## Reflection

*After completing:* What would happen to a pipeline if a CLI tool printed progress messages to stdout instead of stderr? Why is `click.echo(..., err=True)` the right way to write status messages?

## References

- Click documentation: https://click.palletsprojects.com/
- Taschuk & Wilson (2017) Rule 7: "Make it easy to specify configuration settings."